In [ ]:
import json

from src.utils import paths

#data = []
total_length = 0

genres = set()
topics = set()
wikipedia_count = 0
with open(paths.DATA_DIR / "CLASSLA-web.hr.2.0.jsonl") as f:
    i = 0 
    for line in f:
        record = json.loads(line)
        #data.append(record)
        if "text" in record:
            total_length += len(record["text"])
        genres.add(record["genre"])        
        topics.add(record["topic"])    
        if "https://hr.wikipedia.org" in record["url"]:
   
            wikipedia_count +1 
        i += 1
        if i % 5000 == 0:
            print(i)

import pprint

print("Genres:")
pprint.pprint(sorted(genres), compact=False, width=40)
print("\nTopics:")
pprint.pprint(sorted(topics), compact=False, width=40)
print("wikipedia_count", wikipedia_count)


In [ ]:

from datasets import load_dataset

fineweb_edu = load_dataset(
    path=str(paths.DATA_DIR / "fineweb-edu"),
    split="train"  # or adjust depending on your cache structure
)

print("Loaded FineWeb-Edu dataset!")
print(f"Original dataset size: {len(fineweb_edu)}")


In [ ]:

# Select 100k samples randomly and make a subset
fineweb_edu_10k = fineweb_edu.shuffle(seed=42).select(range(10_000))

print("Subsetted FineWeb-Edu dataset (100k samples).")
print(fineweb_edu_10k)


In [ ]:
from src.tokenizers.BPE import BytePairEncoding

tokenizer = BytePairEncoding(3000)

corpus = " ".join(
    [d["text"] for d in data] + [d["text"] for d in fineweb_edu_10k]
)


tokenizer.fit(corpus)

In [ ]:
fineweb_edu_10k[60]["text"]

In [ ]:
import json
from src.utils import paths

classla_path = paths.DATA_DIR / "CLASSLA-web.hr.2.0.jsonl"

with open(classla_path, "rb") as f:
    offsets = []
    while line := f.readline():
        if line.strip():
            offsets.append(f.tell() - len(line))

print(f"{len(offsets)} records")

# Fetch record 1000 directly
with open(classla_path, "rb") as f:
    f.seek(offsets[999])
    record = json.loads(f.readline())

print(record)

In [ ]:

from tqdm import tqdm
import random

random.shuffle(offsets)

with open(classla_path, "rb") as src, open("shuffled.jsonl", "wb") as dst:
    for offset in tqdm(offsets, desc="Shuffling and writing records"):
        src.seek(offset)
        dst.write(src.readline())
   
print(offsets[:5])

In [ ]:
# Build offsets of the shuffled file
shuffled_offsets = []
with open("shuffled.jsonl", "rb") as f:
    while line := f.readline():
        if line.strip():
            shuffled_offsets.append(f.tell() - len(line))

print(shuffled_offsets[:5])

In [ ]:
len(shuffled_offsets)

In [ ]:
max(shuffled_offsets)

In [ ]:
max(offsets)

In [ ]:
# Build (reconstruct) the offsets for the original classla_path file
offsets = []
with open(classla_path, "rb") as f:
    while line := f.readline():
        if line.strip():  # skip empty lines if any
            offsets.append(f.tell() - len(line))

with open(classla_path, "rb") as f:
    f.seek(offsets[-1])
    last_line_orig = f.readline()
    print("Final offset in classla_path:", offsets[-1])
    print("Length of last line in classla_path:", len(last_line_orig))

with open("shuffled.jsonl", "rb") as f:
    f.seek(shuffled_offsets[-1])
    last_line_shuffled = f.readline()
    print("Final offset in shuffled.jsonl:", shuffled_offsets[-1])
    print("Length of last line in shuffled.jsonl:", len(last_line_shuffled))

In [ ]:
import numpy as np

# Save shuffled_offsets to a file for later use
np.save("shuffled_offsets.npy", np.array(shuffled_offsets, dtype=np.int64))


In [ ]:
offsets = np.load("shuffled_offsets.npy")

In [ ]:
offsets[10000]